# SQL Analysis - Pink Slip Data

In [1]:
import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(
    host='localhost',
    dbname='pinks',
    user='postgres',
    password=os.environ.get('DB_PASSWORD')
)

print('Connected to PostgreSQL')
pd.read_sql_query('SELECT COUNT(*) AS total_slips FROM pink_slip', conn)

Connected to PostgreSQL


C:\Users\Collin Xu\AppData\Local\Temp\ipykernel_46176\3838863064.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query('SELECT COUNT(*) AS total_slips FROM pink_slip', conn)


,total_slips
0,89992


## Revenue by Month

In [2]:
pd.read_sql_query('''
    SELECT
        TO_CHAR(date_received, 'YYYY-MM') AS month,
        COUNT(*) AS total_slips,
        ROUND(SUM(total_amount), 2) AS revenue,
        ROUND(AVG(total_amount), 2) AS avg_slip_value
    FROM pink_slip
    GROUP BY month
    ORDER BY month
''', conn)

C:\Users\Collin Xu\AppData\Local\Temp\ipykernel_46176\2546526494.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query('''


,month,total_slips,revenue,avg_slip_value
0,2021-01,1922,44569.0,23.19
1,2021-02,1899,43307.0,22.81
2,2021-03,2539,55706.0,21.94
3,2021-04,3247,70596.0,21.74
4,2021-05,3377,73667.0,21.81
5,2021-06,3288,69812.0,21.23
6,2021-07,2721,57038.0,20.96
7,2021-08,2357,51414.0,21.81
8,2021-09,2271,48850.0,21.51
9,2021-10,2091,45951.0,21.98


## Top 10 Customers by Total Spend

In [3]:
pd.read_sql_query('''
    SELECT
        first_initial || '. ' || last_name AS customer,
        phone,
        COUNT(*) AS visits,
        ROUND(SUM(total_amount)::numeric, 2) AS total_spent,
        ROUND(AVG(total_amount)::numeric, 2) AS avg_per_visit
    FROM pink_slip
    GROUP BY first_initial, last_name, phone
    ORDER BY total_spent DESC
    LIMIT 10
''', conn)

C:\Users\Collin Xu\AppData\Local\Temp\ipykernel_46176\804743648.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query('''


,customer,phone,visits,total_spent,avg_per_visit
0,Z. Smith,(704) 555-3689,23,1568.0,68.17
1,Q. Jones,(704) 555-5822,25,1549.0,61.96
2,S. George,(704) 555-5123,20,1396.0,69.80
3,R. Day,(704) 555-6775,19,1376.0,72.42
4,H. Mann,(704) 555-6213,16,1309.0,81.81
5,D. Horton,(980) 555-2773,20,1280.0,64.00
6,L. Little,(704) 555-9380,22,1225.0,55.68
7,X. Ross,(910) 555-6499,13,1211.0,93.15
8,C. Nunez,(704) 555-9450,17,1191.0,70.06
9,L. Adams,(704) 555-7750,19,1185.0,62.37


## Item Type Breakdown

Revenue and volume by item type.

In [4]:
pd.read_sql_query('''
    SELECT
        i.item_type,
        COUNT(*) AS total_items,
        ROUND(AVG(i.price)::numeric, 2) AS avg_price,
        ROUND(SUM(i.price)::numeric, 2) AS total_revenue,
        ROUND((SUM(i.price) * 100.0 / (SELECT SUM(price) FROM pink_slip_item))::numeric, 1) AS pct_of_revenue
    FROM pink_slip_item i
    INNER JOIN pink_slip s ON i.slip_id = s.id
    GROUP BY i.item_type
    ORDER BY total_revenue DESC
''', conn)

C:\Users\Collin Xu\AppData\Local\Temp\ipykernel_46176\2330947449.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query('''


,item_type,total_items,avg_price,total_revenue,pct_of_revenue
0,Dress,39532,9.67,382348.0,16.4
1,Pants,42925,8.18,351027.0,15.0
2,Jacket,40627,8.55,347160.0,14.9
3,Jeans,38747,8.15,315650.0,13.5
4,Coat,30719,10.01,307400.0,13.2
5,Shirt,33736,6.85,231209.0,9.9
6,Skirt,22493,8.59,193169.0,8.3
7,Shorts,19897,6.86,136557.0,5.8
8,Other,8084,8.82,71299.0,3.1


## Most Common Alterations by Item Type

In [5]:
pd.read_sql_query('''
    SELECT
        item_type,
        work_description,
        COUNT(*) AS times_performed,
        ROUND(AVG(price)::numeric, 2) AS avg_price
    FROM pink_slip_item
    WHERE work_description IS NOT NULL AND work_description != ''
    GROUP BY item_type, work_description
    HAVING COUNT(*) > 10
    ORDER BY item_type, times_performed DESC
''', conn)

C:\Users\Collin Xu\AppData\Local\Temp\ipykernel_46176\696475186.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query('''


,item_type,work_description,times_performed,avg_price
0,Coat,Hem,7588,5.08
1,Coat,Shorten sleeves,6036,6.16
2,Coat,Take in waist,5616,8.18
3,Coat,Repair,4491,9.23
4,Coat,Replace zipper,2592,16.48
...,...,...,...,...
119,Skirt,Minor fix,4042,3.38
120,Skirt,Repair,2254,9.23
121,Skirt,Resize,1859,20.66
122,Skirt,Add lining,1324,25.82


## Customer Retention Analysis

Segmenting customers by visit frequency to see which groups drive the most revenue.

In [6]:
pd.read_sql_query('''
    WITH customer_visits AS (
        SELECT
            phone,
            first_initial || '. ' || last_name AS customer,
            COUNT(*) AS visit_count,
            ROUND(SUM(total_amount)::numeric, 2) AS lifetime_value
        FROM pink_slip
        GROUP BY phone, first_initial, last_name
    )
    SELECT
        CASE
            WHEN visit_count = 1 THEN 'One-time'
            WHEN visit_count BETWEEN 2 AND 4 THEN 'Repeat (2-4)'
            WHEN visit_count BETWEEN 5 AND 9 THEN 'Regular (5-9)'
            ELSE 'VIP (10+)'
        END AS customer_segment,
        COUNT(*) AS num_customers,
        ROUND((COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customer_visits))::numeric, 1) AS pct_of_customers,
        ROUND(SUM(lifetime_value)::numeric, 2) AS total_revenue,
        ROUND((SUM(lifetime_value) * 100.0 / (SELECT SUM(lifetime_value) FROM customer_visits))::numeric, 1) AS pct_of_revenue,
        ROUND(AVG(lifetime_value)::numeric, 2) AS avg_lifetime_value
    FROM customer_visits
    GROUP BY customer_segment
    ORDER BY num_customers DESC
''', conn)

C:\Users\Collin Xu\AppData\Local\Temp\ipykernel_46176\242851538.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query('''


,customer_segment,num_customers,pct_of_customers,total_revenue,pct_of_revenue,avg_lifetime_value
0,Repeat (2-4),9023,42.1,632277.0,27.1,70.07
1,One-time,5717,26.7,143269.0,6.1,25.06
2,Regular (5-9),4346,20.3,757071.0,32.4,174.20
3,VIP (10+),2322,10.8,803202.0,34.4,345.91


## Quarterly Revenue Analysis

Comparing revenue across Q1-Q4 to check for seasonal patterns.

In [7]:
pd.read_sql_query('''
    WITH quarterly_data AS (
        SELECT
            EXTRACT(YEAR FROM date_received)::text AS year,
            CASE
                WHEN EXTRACT(MONTH FROM date_received) IN (1, 2, 3) THEN 'Q1 (Jan-Mar)'
                WHEN EXTRACT(MONTH FROM date_received) IN (4, 5, 6) THEN 'Q2 (Apr-Jun)'
                WHEN EXTRACT(MONTH FROM date_received) IN (7, 8, 9) THEN 'Q3 (Jul-Sep)'
                ELSE 'Q4 (Oct-Dec)'
            END AS quarter,
            total_amount
        FROM pink_slip
    )
    SELECT
        year,
        quarter,
        COUNT(*) AS total_orders,
        ROUND(SUM(total_amount), 2) AS revenue,
        ROUND(AVG(total_amount), 2) AS avg_order_value
    FROM quarterly_data
    GROUP BY year, quarter
    ORDER BY year, quarter
''', conn)

C:\Users\Collin Xu\AppData\Local\Temp\ipykernel_46176\869682627.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query('''


,year,quarter,total_orders,revenue,avg_order_value
0,2021,Q1 (Jan-Mar),6360,143582.0,22.58
1,2021,Q2 (Apr-Jun),9912,214075.0,21.60
2,2021,Q3 (Jul-Sep),7349,157302.0,21.40
3,2021,Q4 (Oct-Dec),6244,139262.0,22.30
4,2022,Q1 (Jan-Mar),6131,167490.0,27.32
5,2022,Q2 (Apr-Jun),9855,256001.0,25.98
6,2022,Q3 (Jul-Sep),7212,189971.0,26.34
7,2022,Q4 (Oct-Dec),6172,167338.0,27.11
8,2023,Q1 (Jan-Mar),6472,193250.0,29.86
9,2023,Q2 (Apr-Jun),10154,295618.0,29.11


## Revenue by Alteration Type

In [8]:
pd.read_sql_query('''
    SELECT
        work_description AS alteration_type,
        COUNT(*) AS times_performed,
        ROUND(SUM(price)::numeric, 2) AS total_revenue,
        ROUND(AVG(price)::numeric, 2) AS avg_price,
        ROUND((SUM(price) * 100.0 / (SELECT SUM(price) FROM pink_slip_item))::numeric, 1) AS pct_of_revenue
    FROM pink_slip_item
    WHERE work_description IS NOT NULL AND work_description != ''
    GROUP BY work_description
    ORDER BY total_revenue DESC
    LIMIT 5
''', conn)

C:\Users\Collin Xu\AppData\Local\Temp\ipykernel_46176\3678846256.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query('''


,alteration_type,times_performed,total_revenue,avg_price,pct_of_revenue
0,Take in waist,51451,421012.0,8.18,18.0
1,Hem,79791,405513.0,5.08,17.4
2,Resize,19188,394042.0,20.54,16.9
3,Repair,29834,276088.0,9.25,11.8
4,Add lining,6835,176220.0,25.78,7.5
